In [8]:
# Load env variables and create API client
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-5"

In [9]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            return block.text

In [ ]:
# Esto esta deprecado en los nuevos modelos

"""
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
"""

In [14]:
# https://platform.claude.com/docs/en/build-with-claude/structured-outputs
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")


response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    output_config={
        "format":{
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "source": {"type": "string"},
                    "detail-type": {"type": "string"},
                },
                "additionalProperties": False,
            }

        }
    }
)

for block in response.content:
        if block.type == "text":
            print(block.text)

{"source":"aws.ec2","detail-type":"EC2 Instance State-change Notification"}
